# Testing and Refactoring Collaborating Objects


```{contents}
:local:
:depth: 2
```


```{index} Test Through the Surface a Client Is Allowed to Use
```



```{index} Test Through the Surface a Client Is Allowed to Use
```

## Test Through the Surface a Client Is Allowed to Use



A collaboration test should establish observable behavior, not reach into another class's private list to force the result. Create two independent orders, add a line to one, and check both totals. If adding to the first changes the second, shared storage has crossed an ownership boundary even if the added line itself has the right amount.

Arrange a state in which a rejected update would be visible. Record the original count and total, attempt the invalid change, and assert that both remain unchanged. Checking only that an exception was thrown can miss a method that mutates state and then throws. A failure policy includes what happened to state, not merely which message appeared.

| Case | Observation to retain |
| --- | --- |
| Two orders share one immutable product | Independent line lists and totals |
| Invalid quantity is rejected | Prior count and total remain intact |
| A snapshot is obtained | The documented outer-copy/sharing policy holds |
| Optional information is absent | The explicit no-information behavior occurs |

Guided challenge: a developer returns the internal line list to make a report easy. Show how a client could change the order without calling its validation methods. Then compare returning a copied outer sequence whose immutable product/line objects remain shared. Explain which mutation path is blocked and why this is not necessarily a deep copy of every reachable object.

For related explanation, see the further reading below.[^development-reading]


A printed total is useful evidence, but one successful order does not establish the contract of an order system. Test the boundaries between the collaborating objects: valid product values, valid line quantities, list ownership, and the effect of rejected operations.

Use the testing workflow from Chapter 6. State an expected result before running the program, arrange the objects, act through public methods, and compare the observed state with that expectation. Small console assertions make the examples self-contained; a durable test project can express the same checks with the course's unit-testing framework.


```{index} Contracts Describe Observable Behavior
```

## Contracts Describe Observable Behavior




```{index} Contracts Describe Observable Behavior
```


A **precondition** describes an input or state required for an operation. A **postcondition** describes what is true after a successful operation. An **invariant** is a condition the model preserves at its public boundaries.

For this order model, products have nonblank identifiers and nonnegative prices; lines have products and positive quantities; the order total is the sum of its lines. An invalid add must leave the existing order unchanged. The total of an empty order is zero. These rules are precise enough to check without inspecting private fields.

Create a boundary table before writing assertions:

| Case | Expected behavior |
| --- | --- |
| Empty order | Zero lines and zero total. |
| Valid quantity one | One line with one unit's amount. |
| Quantity zero | Reject; preserve previous lines and total. |
| Repeated SKU | Separate lines; sum quantities for the SKU. |
| Changed snapshot array | Existing order unchanged. |

A contract also identifies what is outside scope. This model does not reserve warehouse stock, calculate tax, or connect to a database. Do not infer those guarantees from a passing total check.


```{index} Arrange, Act, Assert
```

## Arrange, Act, Assert




```{index} Arrange, Act, Assert
```


The helper below throws if an expectation fails. A normal printed result means the checks completed; an exception points to a failed claim. The test data uses small decimal prices whose expected totals you can calculate independently.

Predict the empty and nonempty totals, then run the complete cell. You can temporarily change the expected value to see how the assertion reports a failure.


In [1]:
using System;
using System.Collections.Generic;

Order order = new Order();
Check(order.Count == 0 && order.Total == 0m, "empty order");
Product notebook = new Product("P100", "Notebook", 6m);
order.Add(notebook, 2);
Check(order.Total == 12m, "two notebooks");
order.Add(new Product("P200", "Pen", 2m), 3);
Check(order.Count == 2 && order.Total == 18m, "mixed order");
Console.WriteLine("3 order checks passed.");

void Check(bool condition, string message)
{
    if (!condition) throw new Exception(message);
}

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}

class OrderLine
{
    public Product Product { get; }
    public int Quantity { get; }
    public decimal LineTotal { get { return Product.UnitPrice * Quantity; } }

    public OrderLine(Product product, int quantity)
    {
        if (product is null) throw new ArgumentNullException(nameof(product));
        if (quantity <= 0) throw new ArgumentOutOfRangeException(nameof(quantity));
        Product = product;
        Quantity = quantity;
    }
}

class Order
{
    private readonly List<OrderLine> lines = new List<OrderLine>();
    public int Count { get { return lines.Count; } }
    public decimal Total
    {
        get
        {
            decimal total = 0m;
            foreach (OrderLine line in lines) total += line.LineTotal;
            return total;
        }
    }
    public void Add(Product product, int quantity)
    {
        OrderLine line = new OrderLine(product, quantity);
        lines.Add(line);
    }
    public int CountUnits(string sku)
    {
        int count = 0;
        foreach (OrderLine line in lines)
            if (line.Product.Sku == sku) count += line.Quantity;
        return count;
    }
    public OrderLine[] GetLines() { return lines.ToArray(); }
}


3 order checks passed.


This is a unit-oriented check of the model, even though several small objects participate. A test framework can provide named tests and richer reports, but the important reasoning is the same. Keep expected values independent: calling `Total` to compute its own expected result would not challenge the implementation.

A passing check is evidence for the exercised cases, not a proof for all inputs. Include cases that would reveal likely defects, such as using only the last line's amount or increasing the line count before validation finishes.


```{index} Rejected Changes Must Preserve State
```

## Rejected Changes Must Preserve State




```{index} Rejected Changes Must Preserve State
```


Failure behavior is part of the contract. Save the line count and total before an invalid add. Catch only the exception type you expect, then check both that rejection occurred and that observable state is unchanged.

Do not catch every exception and print “passed.” A null-reference bug or a failed assertion should not be mistaken for the intended quantity rejection.


In [1]:
using System;
using System.Collections.Generic;

Order order = new Order();
Product notebook = new Product("P100", "Notebook", 6m);
order.Add(notebook, 2);
int countBefore = order.Count;
decimal totalBefore = order.Total;
bool rejected = false;
try { order.Add(notebook, 0); }
catch (ArgumentOutOfRangeException) { rejected = true; }
Console.WriteLine($"Rejected: {rejected}");
Console.WriteLine($"Unchanged: {order.Count == countBefore && order.Total == totalBefore}");

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}

class OrderLine
{
    public Product Product { get; }
    public int Quantity { get; }
    public decimal LineTotal { get { return Product.UnitPrice * Quantity; } }

    public OrderLine(Product product, int quantity)
    {
        if (product is null) throw new ArgumentNullException(nameof(product));
        if (quantity <= 0) throw new ArgumentOutOfRangeException(nameof(quantity));
        Product = product;
        Quantity = quantity;
    }
}

class Order
{
    private readonly List<OrderLine> lines = new List<OrderLine>();
    public int Count { get { return lines.Count; } }
    public decimal Total
    {
        get
        {
            decimal total = 0m;
            foreach (OrderLine line in lines) total += line.LineTotal;
            return total;
        }
    }
    public void Add(Product product, int quantity)
    {
        OrderLine line = new OrderLine(product, quantity);
        lines.Add(line);
    }
    public int CountUnits(string sku)
    {
        int count = 0;
        foreach (OrderLine line in lines)
            if (line.Product.Sku == sku) count += line.Quantity;
        return count;
    }
    public OrderLine[] GetLines() { return lines.ToArray(); }
}


Rejected: True
Unchanged: True


The validation lives in `OrderLine` and completes before the order stores the new line. This ordering makes failure atomic at the model level: the invalid operation changes nothing. Database transactions in the optional appendix solve a related problem across persistent changes, but no database is needed to understand this object-level promise.

Also test two orders sharing one product. Adding a line to the first order must not add it to the second. Shared immutable catalog data is allowed; a shared order list would violate the intended independence.


```{index} Refactor Without Changing the Contract
```

## Refactor Without Changing the Contract




```{index} Refactor Without Changing the Contract
```


**Refactoring** changes internal organization while preserving intended observable behavior. Suppose a console program keeps product names, prices, and quantities in parallel lists. Each insertion must update all three lists in the same order, so a missing update can silently pair the wrong price with a product.

First capture a few expected orders as checks. Introduce `Product` and `OrderLine`, move line calculations into the line class, and move the line collection into `Order`. Keep the public task the same: add requested products and obtain the correct total. Run the checks after each small change.

Do not combine a refactor with an unexplained change in business policy. Merging duplicate lines, rounding tax, or changing case-sensitive SKU matching changes behavior. Make that a separate requirement with new expectations.

A useful review asks who owns each rule. The model should not read from `Console` just to calculate a total. The console client should not reach into the list to repair invalid quantities. Separating those jobs lets tests use fixed inputs without simulating keyboard interaction.


```{index} Prepare a Durable Multi-Class Project
```

## Prepare a Durable Multi-Class Project




```{index} Prepare a Durable Multi-Class Project
```


Use the reference project in `demos/11_object_design/object_design/` to inspect a saved application. Its default run prints an order summary; `--check` runs boundary checks. The same model works without the interactive console because its methods receive ordinary values.

```sh
dotnet run --project demos/11_object_design/object_design
dotnet run --project demos/11_object_design/object_design -- --check
```

For your own project, use `Product.cs`, `OrderLine.cs`, `Order.cs`, and `Program.cs`, or an equally clear organization. Write a README that states identifier equality, duplicate-line behavior, price history, invalid-input handling, and how to run the checks. These are design decisions a teammate must be able to discover.

After these foundations, Chapter 12 introduces interfaces and inheritance for situations in which clients must work with several kinds of implementation. Keep ordinary composition when one concrete class already fits the problem.


```{index} Practice: Challenge the Ownership Rules
```

## Practice: Challenge the Ownership Rules




```{index} Practice: Challenge the Ownership Rules
```


Write checks for two independent orders sharing one product, then replace an element in a returned snapshot. Show that both ownership rules still hold.


In [ ]:
using System;
using System.Collections.Generic;

Product notebook = new Product("P100", "Notebook", 6m);
Order first = new Order();
Order second = new Order();
first.Add(notebook, 2);
// Your code starts here.
/* TODO: check the second order. */
// Your code ends here.
OrderLine[] snapshot = first.GetLines();
snapshot[0] = new OrderLine(notebook, 50);
/* TODO: check the first order after editing the snapshot. */
Console.WriteLine("Order independence and snapshot checks passed.");

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}

class OrderLine
{
    public Product Product { get; }
    public int Quantity { get; }
    public decimal LineTotal { get { return Product.UnitPrice * Quantity; } }

    public OrderLine(Product product, int quantity)
    {
        if (product is null) throw new ArgumentNullException(nameof(product));
        if (quantity <= 0) throw new ArgumentOutOfRangeException(nameof(quantity));
        Product = product;
        Quantity = quantity;
    }
}

class Order
{
    private readonly List<OrderLine> lines = new List<OrderLine>();
    public int Count { get { return lines.Count; } }
    public decimal Total
    {
        get
        {
            decimal total = 0m;
            foreach (OrderLine line in lines) total += line.LineTotal;
            return total;
        }
    }
    public void Add(Product product, int quantity)
    {
        OrderLine line = new OrderLine(product, quantity);
        lines.Add(line);
    }
    public int CountUnits(string sku)
    {
        int count = 0;
        foreach (OrderLine line in lines)
            if (line.Product.Sku == sku) count += line.Quantity;
        return count;
    }
    public OrderLine[] GetLines() { return lines.ToArray(); }
}


In [1]:
using System;
using System.Collections.Generic;

Product notebook = new Product("P100", "Notebook", 6m);
Order first = new Order();
Order second = new Order();
first.Add(notebook, 2);
if (second.Count != 0 || second.Total != 0m) throw new Exception("Orders share state.");
OrderLine[] snapshot = first.GetLines();
snapshot[0] = new OrderLine(notebook, 50);
if (first.Total != 12m) throw new Exception("Snapshot leaked storage.");
Console.WriteLine("Order independence and snapshot checks passed.");

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}

class OrderLine
{
    public Product Product { get; }
    public int Quantity { get; }
    public decimal LineTotal { get { return Product.UnitPrice * Quantity; } }

    public OrderLine(Product product, int quantity)
    {
        if (product is null) throw new ArgumentNullException(nameof(product));
        if (quantity <= 0) throw new ArgumentOutOfRangeException(nameof(quantity));
        Product = product;
        Quantity = quantity;
    }
}

class Order
{
    private readonly List<OrderLine> lines = new List<OrderLine>();
    public int Count { get { return lines.Count; } }
    public decimal Total
    {
        get
        {
            decimal total = 0m;
            foreach (OrderLine line in lines) total += line.LineTotal;
            return total;
        }
    }
    public void Add(Product product, int quantity)
    {
        OrderLine line = new OrderLine(product, quantity);
        lines.Add(line);
    }
    public int CountUnits(string sku)
    {
        int count = 0;
        foreach (OrderLine line in lines)
            if (line.Product.Sku == sku) count += line.Quantity;
        return count;
    }
    public OrderLine[] GetLines() { return lines.ToArray(); }
}


Order independence and snapshot checks passed.


```{rubric} Footnotes
```


[^1103_testing_collaboration-1]: Console assertions here are ordinary C# condition checks followed by exceptions. They are deliberately small so each browser example is complete; Chapter 6 explains the larger testing workflow.


[^development-reading]: Further reading: [Microsoft, C# Record Types](https://learn.microsoft.com/en-us/dotnet/csharp/fundamentals/types/records). The worked business scenarios and guided reasoning here are original teaching examples.
